# Mutual Fund Analytics Platform
## Day 2 - Data Cleaning & SQLite Database

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from sqlalchemy import create_engine

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
fund_master = pd.read_csv("../data/raw/01_fund_master.csv")
nav_history = pd.read_csv("../data/raw/02_nav_history.csv")
aum = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")
sip = pd.read_csv("../data/raw/04_monthly_sip_inflows.csv")
transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

print("All files loaded successfully")

All files loaded successfully


In [3]:
print("Fund Master:", fund_master.shape)
print("NAV History:", nav_history.shape)
print("AUM:", aum.shape)
print("SIP:", sip.shape)
print("Transactions:", transactions.shape)
print("Performance:", performance.shape)

Fund Master: (40, 15)
NAV History: (46000, 3)
AUM: (90, 5)
SIP: (48, 6)
Transactions: (32778, 13)
Performance: (40, 19)


In [4]:
nav_history["date"] = pd.to_datetime(nav_history["date"], errors="coerce")
nav_history = nav_history.sort_values(["amfi_code","date"])
nav_history["nav"] = nav_history.groupby("amfi_code")["nav"].ffill()
nav_history = nav_history[nav_history["nav"] > 0]
nav_history = nav_history.drop_duplicates()
print("NAV cleaned")
print(nav_history.shape)

NAV cleaned
(46000, 3)


In [5]:
nav_history.isnull().sum()

amfi_code    0
date         0
nav          0
dtype: int64

In [6]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"], errors="coerce")
transactions["transaction_type"] = transactions["transaction_type"].astype(str).str.strip().str.title()
transactions = transactions[transactions["amount_inr"] > 0]
transactions = transactions.drop_duplicates()
print("Transactions cleaned")
print(transactions.shape)

Transactions cleaned
(32778, 13)


In [7]:
transactions["kyc_status"].unique()

<StringArray>
['Verified', 'Pending']
Length: 2, dtype: str

In [8]:
return_cols = ["return_1yr_pct","return_3yr_pct","return_5yr_pct","benchmark_3yr_pct"]

for col in return_cols:
    performance[col] = pd.to_numeric(performance[col], errors="coerce")

performance = performance[(performance["expense_ratio_pct"] >= 0.1) & (performance["expense_ratio_pct"] <= 2.5)]
performance = performance.drop_duplicates()

print("Performance cleaned")
print(performance.shape)

Performance cleaned
(40, 19)


In [9]:
performance[["scheme_name","expense_ratio_pct"]].head()

,scheme_name,expense_ratio_pct
0,SBI Bluechip Fund - Regular Plan - Growth,1.54
1,SBI Bluechip Fund - Direct Plan - Growth,0.66
2,SBI Small Cap Fund - Regular Plan - Growth,1.43
3,SBI Small Cap Fund - Direct Plan - Growth,0.72
4,SBI Magnum Gilt Fund - Regular Plan - Growth,0.77


In [10]:
nav_history.to_csv("../data/processed/cleaned_nav_history.csv", index=False)
transactions.to_csv("../data/processed/cleaned_transactions.csv", index=False)
performance.to_csv("../data/processed/cleaned_performance.csv", index=False)

print("Processed files saved")

Processed files saved


In [11]:
engine = create_engine("sqlite:///../bluestock_mf.db")
print("Database Connected")

Database Connected


In [12]:
fund_master.to_sql("dim_fund", engine, if_exists="replace", index=False)
nav_history.to_sql("fact_nav", engine, if_exists="replace", index=False)
transactions.to_sql("fact_transactions", engine, if_exists="replace", index=False)
performance.to_sql("fact_performance", engine, if_exists="replace", index=False)
aum.to_sql("fact_aum", engine, if_exists="replace", index=False)
sip.to_sql("fact_sip", engine, if_exists="replace", index=False)

print("SQLite tables loaded")

SQLite tables loaded


In [13]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", engine)

,name
0,dim_date
1,fact_portfolio
2,fact_sip_inflows
3,fact_category_inflows
4,fact_folio_count
5,fact_benchmark
6,dim_fund
7,fact_nav
8,fact_transactions
9,fact_performance


In [14]:
tables = ["dim_fund","fact_nav","fact_transactions","fact_performance","fact_aum","fact_sip"]
for table in tables:
    query = f"SELECT COUNT(*) AS rows FROM {table}"
    print(table)
    print(pd.read_sql(query, engine))

dim_fund
   rows
0    40
fact_nav
    rows
0  46000
fact_transactions
    rows
0  32778
fact_performance
   rows
0    40
fact_aum
   rows
0    90
fact_sip
   rows
0    48


In [15]:
query = """SELECT fund_house, MAX(aum_crore) AS total_aum FROM fact_aum GROUP BY fund_house ORDER BY total_aum DESC LIMIT 5;"""
pd.read_sql(query, engine)

,fund_house,total_aum
0,SBI Mutual Fund,1250000
1,ICICI Prudential MF,1074000
2,HDFC Mutual Fund,930000
3,Nippon India MF,700000
4,Kotak Mahindra MF,580000


In [16]:
query = """SELECT amfi_code, ROUND(AVG(nav),2) AS avg_nav FROM fact_nav GROUP BY amfi_code ORDER BY avg_nav DESC;"""
pd.read_sql(query, engine)

,amfi_code,avg_nav
0,120844,3705.88
1,125497,797.45
2,100016,566.19
3,149322,439.18
4,101206,435.98
5,101208,358.39
6,120841,355.08
7,120507,334.08
8,120505,284.00
9,120506,274.32


In [17]:
query = """SELECT state, COUNT(*) AS transactions, SUM(amount_inr) AS total_amount FROM fact_transactions GROUP BY state ORDER BY total_amount DESC;"""
pd.read_sql(query, engine)

,state,transactions,total_amount
0,Punjab,2965,315780459
1,Tamil Nadu,2806,315177237
2,Madhya Pradesh,2931,308312493
3,Rajasthan,2577,298645822
4,Gujarat,2780,298358940
5,West Bengal,2748,297182514
6,Telangana,2718,290219284
7,Delhi,2677,289633404
8,Uttar Pradesh,2695,285368873
9,Haryana,2736,279634354


In [18]:
query = """SELECT scheme_name, expense_ratio_pct FROM fact_performance WHERE expense_ratio_pct < 1;"""
pd.read_sql(query, engine)

,scheme_name,expense_ratio_pct
0,SBI Bluechip Fund - Direct Plan - Growth,0.66
1,SBI Small Cap Fund - Direct Plan - Growth,0.72
2,SBI Magnum Gilt Fund - Regular Plan - Growth,0.77
3,HDFC Top 100 Fund - Direct Plan - Growth,0.92
4,HDFC Mid-Cap Opportunities Fund - Direct - Growth,0.78
5,HDFC Short Term Debt Fund - Regular - Growth,0.56
6,ICICI Pru Bluechip Fund - Direct - Growth,0.80
7,ICICI Pru Liquid Fund - Regular - Growth,0.74
8,Nippon India Large Cap Fund - Direct - Growth,0.72
9,Nippon India ETF Nifty 50 BeES,0.89


In [19]:
query = """SELECT scheme_name, return_5yr_pct FROM fact_performance ORDER BY return_5yr_pct DESC LIMIT 10;"""
pd.read_sql(query, engine)

,scheme_name,return_5yr_pct
0,ABSL Small Cap Fund - Regular - Growth,23.80
1,Axis Small Cap Fund - Regular - Growth,22.62
2,Nippon India Small Cap Fund - Regular - Growth,21.88
3,SBI Small Cap Fund - Direct Plan - Growth,21.82
4,SBI Small Cap Fund - Regular Plan - Growth,20.67
5,DSP Small Cap Fund - Regular - Growth,20.61
6,DSP Midcap Fund - Regular - Growth,19.00
7,Axis Midcap Fund - Regular - Growth,18.94
8,Kotak Emerging Equity Fund - Regular - Growth,17.75
9,HDFC Mid-Cap Opportunities Fund - Regular - Gr...,17.69


In [20]:
query = """SELECT risk_grade, COUNT(*) AS funds FROM fact_performance GROUP BY risk_grade;"""
pd.read_sql(query, engine)

,risk_grade,funds
0,High,8
1,Low,6
2,Moderate,16
3,Moderately High,4
4,Very High,6


In [21]:
query = """SELECT transaction_type, COUNT(*) AS total FROM fact_transactions GROUP BY transaction_type;"""
pd.read_sql(query, engine)

,transaction_type,total
0,Lumpsum,8095
1,Redemption,4967
2,Sip,19716


In [22]:
query = """SELECT category, ROUND(AVG(return_3yr_pct),2) AS avg_return FROM fact_performance GROUP BY category;"""
pd.read_sql(query, engine)

,category,avg_return
0,ELSS,13.58
1,Flexi Cap,15.50
2,Gilt,5.69
3,Index,12.10
4,Index/ETF,11.77
5,Large & Mid Cap,14.56
6,Large Cap,12.99
7,Liquid,6.33
8,Mid Cap,16.59
9,Short Duration,7.37


In [23]:
query = """SELECT investor_id, SUM(amount_inr) AS total_investment FROM fact_transactions GROUP BY investor_id ORDER BY total_investment DESC LIMIT 10;"""
pd.read_sql(query, engine)

,investor_id,total_investment
0,INV003288,3480685
1,INV001999,3179916
2,INV004605,2882975
3,INV000477,2846388
4,INV001692,2751086
5,INV002039,2713616
6,INV001771,2695710
7,INV002553,2668557
8,INV002707,2651584
9,INV002245,2623784


In [24]:
query = """SELECT state, ROUND(AVG(amount_inr),2) AS avg_amount FROM fact_transactions GROUP BY state ORDER BY avg_amount DESC;"""
pd.read_sql(query, engine)

,state,avg_amount
0,Rajasthan,115888.95
1,Tamil Nadu,112322.61
2,Delhi,108193.28
3,West Bengal,108145.02
4,Gujarat,107323.36
5,Maharashtra,106780.30
6,Telangana,106776.78
7,Punjab,106502.68
8,Uttar Pradesh,105888.26
9,Madhya Pradesh,105190.21


# Day 2 Completed

Deliverables:
- cleaned_nav_history.csv
- cleaned_transactions.csv
- cleaned_performance.csv
- bluestock_mf.db
- schema.sql
- queries.sql
- data_dictionary.md
